In [0]:
def get_baseline_metrics():

    result = spark.sql("""
        SELECT 
            SUM(revenue) AS baseline_revenue,
            SUM(units_sold) AS baseline_units
        FROM cpg_sales_silver
    """).collect()[0]

    return float(result["baseline_revenue"]), float(result["baseline_units"])

In [0]:
def simulate_supply_shortage(shortage_percent):

    baseline_revenue, baseline_units = get_baseline_metrics()

    reduction_factor = 1 - shortage_percent / 100

    result = spark.sql(f"""
        SELECT 
            SUM(units_sold * {reduction_factor} * price) AS projected_revenue,
            SUM(units_sold * {reduction_factor}) AS projected_units
        FROM cpg_sales_silver
    """).collect()[0]

    projected_revenue = float(result["projected_revenue"])
    projected_units = float(result["projected_units"])

    impact = ((projected_revenue - baseline_revenue) / baseline_revenue) * 100

    return {
        "simulation_type": "Supply Shortage",
        "shortage_percent": shortage_percent,
        "baseline_revenue": round(baseline_revenue, 2),
        "projected_revenue": round(projected_revenue, 2),
        "revenue_impact_percent": round(impact, 2),
        "baseline_units": baseline_units,
        "projected_units": projected_units
    }


In [0]:
def simulate_price_increase(percent):

    baseline_revenue, baseline_units = get_baseline_metrics()

    price_factor = 1 + percent / 100
    demand_factor = 1 - (percent / 200)  # simple elasticity logic

    result = spark.sql(f"""
        SELECT 
            SUM(units_sold * {demand_factor} * price * {price_factor}) AS projected_revenue,
            SUM(units_sold * {demand_factor}) AS projected_units
        FROM cpg_sales_silver
    """).collect()[0]

    projected_revenue = float(result["projected_revenue"])
    projected_units = float(result["projected_units"])

    impact = ((projected_revenue - baseline_revenue) / baseline_revenue) * 100

    return {
        "simulation_type": "Price Increase",
        "price_increase_percent": percent,
        "baseline_revenue": round(baseline_revenue, 2),
        "projected_revenue": round(projected_revenue, 2),
        "revenue_impact_percent": round(impact, 2),
        "baseline_units": baseline_units,
        "projected_units": projected_units
    }


In [0]:
def simulate_promo_uplift(uplift_percent):

    baseline_revenue, baseline_units = get_baseline_metrics()

    uplift_factor = 1 + uplift_percent / 100

    result = spark.sql(f"""
        SELECT 
            SUM(
                CASE 
                    WHEN promo_flag = 1 
                    THEN units_sold * {uplift_factor} * price
                    ELSE units_sold * price
                END
            ) AS projected_revenue,
            SUM(
                CASE 
                    WHEN promo_flag = 1 
                    THEN units_sold * {uplift_factor}
                    ELSE units_sold
                END
            ) AS projected_units
        FROM cpg_sales_silver
    """).collect()[0]

    projected_revenue = float(result["projected_revenue"])
    projected_units = float(result["projected_units"])

    impact = ((projected_revenue - baseline_revenue) / baseline_revenue) * 100

    return {
        "simulation_type": "Promo Uplift",
        "uplift_percent": uplift_percent,
        "baseline_revenue": round(baseline_revenue, 2),
        "projected_revenue": round(projected_revenue, 2),
        "revenue_impact_percent": round(impact, 2),
        "baseline_units": baseline_units,
        "projected_units": projected_units
    }


In [0]:
def simulate_combined_strategy(price_percent, promo_percent):

    baseline_revenue, baseline_units = get_baseline_metrics()

    price_factor = 1 + price_percent / 100
    promo_factor = 1 + promo_percent / 100

    result = spark.sql(f"""
        SELECT 
            SUM(
                CASE 
                    WHEN promo_flag = 1 
                    THEN units_sold * {promo_factor} * price * {price_factor}
                    ELSE units_sold * price * {price_factor}
                END
            ) AS projected_revenue
        FROM cpg_sales_Silver
    """).collect()[0]

    projected_revenue = float(result["projected_revenue"])

    impact = ((projected_revenue - baseline_revenue) / baseline_revenue) * 100

    return {
        "simulation_type": "Combined Strategy",
        "price_increase_percent": price_percent,
        "promo_uplift_percent": promo_percent,
        "baseline_revenue": round(baseline_revenue, 2),
        "projected_revenue": round(projected_revenue, 2),
        "revenue_impact_percent": round(impact, 2)
    }


In [0]:
spark.table("cpg_sales_silver").printSchema()

root
 |-- date: date (nullable = true)
 |-- store_id: long (nullable = true)
 |-- store_region: string (nullable = true)
 |-- sku_id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- units_sold: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- promo_flag: long (nullable = true)
 |-- promo_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- inventory_level: long (nullable = true)
 |-- store_size: string (nullable = true)
 |-- holiday_flag: long (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- revenue_per_unit: double (nullable = true)
 |-- is_promo: boolean (nullable = true)

